# DSE 230: DataFrame Hands-On - Solution Notebook

## Common PySpark DataFrame Operations

--- 

Remember: when in doubt, read the documentation first. It's always helpful to search for the class that you're trying to work with, e.g. pyspark.sql.DataFrame. 

Spark DataFrame Guide:  https://spark.apache.org/docs/latest/sql-programming-guide.html

PySpark API Documentation: https://spark.apache.org/docs/latest/api/python/index.html


In [1]:
# Suppress native-hadoop warning
!sed -i '$a\# Add the line for suppressing the NativeCodeLoader warning \nlog4j.logger.org.apache.hadoop.util.NativeCodeLoader=ERROR,console' /$HADOOP_HOME/etc/hadoop/log4j.properties

In [2]:
# initialize Spark

import pyspark
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import *

conf = pyspark.SparkConf().setAll([('spark.master', 'local[2]'),
                                   ('spark.app.name', 'PySpark DataFrame Demo')])
spark = SparkSession.builder.config(conf=conf).getOrCreate()

print (pyspark.version.__version__)

4.1.1


---
## Employee example to show common DataFrame operations

### Create Employee DataFrame

In [3]:
Employee = Row("name", "dept", "state", "salary")
employee1 = Employee('James', 'Sales', 'CA', 100000)
employee2 = Employee('Mary', 'Finance', 'CA', 120000)
employee3 = Employee('Jane', 'Sales', 'WA', 160000)
employees = [employee1, employee2, employee3]
employeesDF = spark.createDataFrame(employees)
employeesDF.show()

+-----+-------+-----+------+
| name|   dept|state|salary|
+-----+-------+-----+------+
|James|  Sales|   CA|100000|
| Mary|Finance|   CA|120000|
| Jane|  Sales|   WA|160000|
+-----+-------+-----+------+



### Check type of employeesDF

In [4]:
type(employeesDF)

pyspark.sql.classic.dataframe.DataFrame

### Count number of employees

In [5]:
employeesDF.count()

3

### Display schema

In [6]:
employeesDF.printSchema()

root
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- state: string (nullable = true)
 |-- salary: long (nullable = true)



### Show contents

Show contents of employeesDF

In [7]:
employeesDF.show()

+-----+-------+-----+------+
| name|   dept|state|salary|
+-----+-------+-----+------+
|James|  Sales|   CA|100000|
| Mary|Finance|   CA|120000|
| Jane|  Sales|   WA|160000|
+-----+-------+-----+------+



How is this different from the previous cell?

In [8]:
employeesDF.select("name").show()

+-----+
| name|
+-----+
|James|
| Mary|
| Jane|
+-----+



What does this return?

In [9]:
employeesDF.show(1)

+-----+-----+-----+------+
| name| dept|state|salary|
+-----+-----+-----+------+
|James|Sales|   CA|100000|
+-----+-----+-----+------+
only showing top 1 row


### Get summary statistics

In [10]:
employeesDF.describe().show()

+-------+-----+-------+-----+------------------+
|summary| name|   dept|state|            salary|
+-------+-----+-------+-----+------------------+
|  count|    3|      3|    3|                 3|
|   mean| NULL|   NULL| NULL|126666.66666666667|
| stddev| NULL|   NULL| NULL|30550.504633038934|
|    min|James|Finance|   CA|            100000|
|    max| Mary|  Sales|   WA|            160000|
+-------+-----+-------+-----+------------------+



### Find highest salary by state
Group employees by state, then find max salary

In [11]:
employeesDF.show()

+-----+-------+-----+------+
| name|   dept|state|salary|
+-----+-------+-----+------+
|James|  Sales|   CA|100000|
| Mary|Finance|   CA|120000|
| Jane|  Sales|   WA|160000|
+-----+-------+-----+------+



In [12]:
employeesDF.groupBy('state').max('salary').show()

+-----+-----------+
|state|max(salary)|
+-----+-----------+
|   CA|     120000|
|   WA|     160000|
+-----+-----------+



### Count employees by department
Group employees by department, then find count.

In [13]:
employeesDF.groupBy('dept').count().show()

+-------+-----+
|   dept|count|
+-------+-----+
|  Sales|    2|
|Finance|    1|
+-------+-----+



### Filter
Get all employees in California

In [14]:
employeesDF.filter(employeesDF.state=='CA').show()

+-----+-------+-----+------+
| name|   dept|state|salary|
+-----+-------+-----+------+
|James|  Sales|   CA|100000|
| Mary|Finance|   CA|120000|
+-----+-------+-----+------+



In [15]:
employeesDF.filter(col("state") == "CA").show()

+-----+-------+-----+------+
| name|   dept|state|salary|
+-----+-------+-----+------+
|James|  Sales|   CA|100000|
| Mary|Finance|   CA|120000|
+-----+-------+-----+------+



### Sort by name

Sort by name alphabetically, and save in a new DataFrame. Then display the new DataFrame.

In [16]:
employees_sortedDF = employeesDF.sort('name', ascending=True)

In [17]:
employees_sortedDF.show()

+-----+-------+-----+------+
| name|   dept|state|salary|
+-----+-------+-----+------+
|James|  Sales|   CA|100000|
| Jane|  Sales|   WA|160000|
| Mary|Finance|   CA|120000|
+-----+-------+-----+------+



### Save sorted DataFrame to file on local system
Fill in sorted DataFrame name, and file name.  Remember that you need "file://\<path\>/\<filename\>" when writing to the local file system.

In [18]:
# Print the working directory
!pwd

/home/work/S2-Spark-EDA/Demo2A-Spark-DataFrame


In [19]:
employees_sortedDF.coalesce(1).\
write.csv("file:///home/work/S2-Spark-EDA/Demo2A-Spark-DataFrame/employees_sorted.csv", 
          header=True, mode="overwrite")

----
## Sentence example 

### Create DataFrame with sentences

In [20]:
sent_0 = Row(value='This is a sentence')
sent_1 = Row(value='This is another sentence')
sentences = [sent_0, sent_1]
sentenceDF = spark.createDataFrame(sentences)
sentenceDF.show()

+--------------------+
|               value|
+--------------------+
|  This is a sentence|
|This is another s...|
+--------------------+



### Examples to show split() and explode()

Split sentences on space, and rename the resulting column to 'csv'. Save the results in a new DataFrame named 'wordsDF1'.

In [21]:
sentenceDF.select(split("value"," ")).show()

+--------------------+
| split(value,  , -1)|
+--------------------+
|[This, is, a, sen...|
|[This, is, anothe...|
+--------------------+



In [22]:
wordsDF1 = sentenceDF.select(split("value"," ").alias("csv"))
wordsDF1.show()

+--------------------+
|                 csv|
+--------------------+
|[This, is, a, sen...|
|[This, is, anothe...|
+--------------------+



Map each column in wordsDF1 to a separate row and rename the column to 'word'. Hint: Use 'explode'.

Put the results in a new DataFrame named 'wordsDF2'.

In [23]:
wordsDF2 = wordsDF1.select(explode("csv").alias("word"))
wordsDF2.show()

+--------+
|    word|
+--------+
|    This|
|      is|
|       a|
|sentence|
|    This|
|      is|
| another|
|sentence|
+--------+



Filter this new DataFrame to return rows containing the word 'This'.

In [24]:
wordsDF2.filter(col("word") == "This").show()

+----+
|word|
+----+
|This|
|This|
+----+



### Stop Spark session

In [25]:
spark.stop()